In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
import ta

# 1. Tải dữ liệu mới nhất (60 phiên gần nhất)
def get_latest_data(ticker):
    # Đọc trực tiếp từ file CSV bạn đã lưu (nó chính là dữ liệu mới nhất bạn vừa tải)
    df = pd.read_csv('VCB_data.csv', index_col=0, parse_dates=True)
    # Đảm bảo dữ liệu được xử lý giống hệt lúc training
    # (Nếu bạn cần lấy dữ liệu cập nhật hơn nữa, hãy dùng hàm này)
    return df[['Close', 'RSI', 'Volume']].tail(60)
# 2. Dự báo
def predict_next_day(ticker):
    # Load model và scaler đã lưu
    model = load_model('vcb_lstm_model.h5')
    scaler = joblib.load('scaler.pkl')
    
    # Chuẩn bị dữ liệu
    df_latest = get_latest_data(ticker)
    scaled_data = scaler.transform(df_latest)
    
    # Chuyển về dạng 3D (1, 60, 3)
    X_input = np.array([scaled_data])
    
    # Dự báo
    predicted_scaled = model.predict(X_input)
    
    # Đảo ngược chuẩn hóa (Inverse Transform)
    # Tạo mảng giả (dummy) 3 cột
    dummy = np.zeros((1, 3))
    dummy[0, 0] = predicted_scaled[0, 0]
    
    predicted_price = scaler.inverse_transform(dummy)[0, 0]
    return predicted_price

# Thực thi
ticker = "VCB.VN"
price = predict_next_day(ticker)
print(f"Giá dự báo cho phiên giao dịch tiếp theo của {ticker} là: {price:,.0f} VND")

C:\Users\Admin\AppData\Local\Temp\ipykernel_13280\794677698.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('VCB_data.csv', index_col=0, parse_dates=True)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step
Giá dự báo cho phiên giao dịch tiếp theo của VCB.VN là: 61,666 VND
